# **Eksperimen UDAO-LASFNet pada Dataset DroneVehicle (RGB-IR)**
Notebook ini menjalankan pipeline eksperimen **UDAO-LASFNet** (*Uncertainty-Driven Deformable Alignment & Differential Feature Attention Transformation*) dari repositori `Allezan/UDAO-LASFNet` pada dataset **DroneVehicle (5 Kelas: `car`, `truck`, `bus`, `van`, `freight car`)**:
1. **UDAO-ASFF** (Layer 10, $P_3/8$): Penyelarasan spasial lintas-modalitas berbasis `deform_conv2d` dengan *learnable offset scale* $\gamma$.
2. **D-FATM** (Layer 24, Neck $P_3/8$): Peningkatan ketajaman tepi frekuensi tinggi (*Sobel + 8-neighbor Laplacian*).
3. **Auxiliary Boundary Loss (BAB)**: Supervisi batas objek berbasis *GT Bounding-Box Masking* ($M_{gt}$) yang hanya aktif saat *training* (**0 ms *inference overhead***).

### **LANGKAH 1: Inisialisasi Environment, Google Drive, & Clone Repositori `Allezan/UDAO-LASFNet`**

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/UDAO_LASFNet_DroneVehicle_Run"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Folder target di Google Drive siap: {SAVE_DIR}")

%cd /content
!rm -rf /content/UDAO-LASFNet
!git clone -b udao-dfatm https://github.com/Allezan/UDAO-LASFNet.git
%cd /content/UDAO-LASFNet

!pip install -q "scikit-image>=0.24.0" xlsxwriter huggingface_hub thop pycocotools ultralytics albumentations timm einops prettytable gitpython
print("Instalasi dependensi & clone repositori Allezan/UDAO-LASFNet selesai!")

### **LANGKAH 2: Unduh Dataset McCheng/DroneVehicle dari Hugging Face**

In [ ]:
import os
import subprocess
from huggingface_hub import snapshot_download

target_dir = "/content/UDAO-LASFNet/dataset/dronevehicle_new"
os.makedirs(target_dir, exist_ok=True)

print("Mengunduh dataset dari Hugging Face...")
snapshot_download(repo_id="McCheng/DroneVehicle", repo_type="dataset", local_dir=target_dir)

for zip_name in ['train.zip', 'val.zip', 'test.zip']:
    zip_path = os.path.join(target_dir, zip_name)
    if os.path.exists(zip_path):
        print(f"Mengekstrak {zip_name}...")
        subprocess.run(["unzip", "-q", "-o", zip_path, "-d", target_dir])

print("Ekstraksi dataset selesai!")

### **LANGKAH 3: Penataan Struktur Direktori & Pembersihan File Bermasalah**

In [ ]:
import os

target_dir = "/content/UDAO-LASFNet/dataset/dronevehicle_new"

for mode in ['train', 'val', 'test']:
    os.makedirs(f"{target_dir}/data_rgb/{mode}/labels", exist_ok=True)
    os.makedirs(f"{target_dir}/data_ir/{mode}/labels", exist_ok=True)
    os.makedirs(f"{target_dir}/data_rgb/{mode}/images", exist_ok=True)
    os.makedirs(f"{target_dir}/data_ir/{mode}/images", exist_ok=True)

print("Menata letak file gambar dan label...")
for mode in ['train', 'val', 'test']:
    !mv {target_dir}/{mode}/{mode}img/* {target_dir}/data_rgb/{mode}/images/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}imgr/* {target_dir}/data_ir/{mode}/images/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}label/* {target_dir}/data_rgb/{mode}/labels/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}labelr/* {target_dir}/data_ir/{mode}/labels/ 2>/dev/null || true

problematic_files = ["04895", "06463", "14048", "12900", "05001", "09012", "04991", "16009", "14035"]

print("\n--- Menghapus File Bermasalah secara Manual ---")
for base_name in problematic_files:
    for mode in ['train', 'val', 'test']:
        paths_to_remove = [
            f"data_rgb/{mode}/images/{base_name}.jpg",
            f"data_ir/{mode}/images/{base_name}.jpg",
            f"data_rgb/{mode}/labels/{base_name}.txt",
            f"data_ir/{mode}/labels/{base_name}.txt",
            f"data_rgb/{mode}/labels/{base_name}.xml",
            f"data_ir/{mode}/labels/{base_name}.xml"
        ]
        for p in paths_to_remove:
            full_path = os.path.join(target_dir, p)
            if os.path.exists(full_path):
                os.remove(full_path)
                print(f"Dihapus: {p}")

print("Pembersihan file manual selesai!")

### **LANGKAH 4: Konversi Format XML DroneVehicle ke YOLO TXT (5 Kelas: Termasuk `freight car` / `feright_car`)**

In [ ]:
import os
import xml.etree.ElementTree as ET
import glob

def convert_dronevehicle_xml_to_yolo(xml_dir, txt_dir, width=840, height=712):
    if not os.path.exists(xml_dir):
        return
    os.makedirs(txt_dir, exist_ok=True)
    class_map = {
        'car': 0,
        'truck': 1,
        'bus': 2,
        'van': 3,
        'freight car': 4,
        'freight_car': 4,
        'feright car': 4,
        'feright_car': 4,
        'feright': 4
    }
    xml_files = glob.glob(os.path.join(xml_dir, "*.xml"))

    for xml_file in xml_files:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        filename = os.path.splitext(os.path.basename(xml_file))[0]
        txt_file_path = os.path.join(txt_dir, f"{filename}.txt")

        yolo_lines = []
        for obj in root.findall('object'):
            name = obj.find('name').text.strip().lower()
            if name not in class_map:
                continue
            class_id = class_map[name]
            polygon = obj.find('polygon')
            bndbox = obj.find('bndbox')
            if polygon is not None:
                try:
                    xs = [float(polygon.find(f'x{i}').text) for i in range(1, 5)]
                    ys = [float(polygon.find(f'y{i}').text) for i in range(1, 5)]
                    xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
                    x_center = ((xmin + xmax) / 2.0) / width
                    y_center = ((ymin + ymax) / 2.0) / height
                    box_w = (xmax - xmin) / width
                    box_h = (ymax - ymin) / height
                    yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")
                except:
                    continue
            elif bndbox is not None:
                try:
                    xmin = float(bndbox.find('xmin').text)
                    ymin = float(bndbox.find('ymin').text)
                    xmax = float(bndbox.find('xmax').text)
                    ymax = float(bndbox.find('ymax').text)
                    x_center = ((xmin + xmax) / 2.0) / width
                    y_center = ((ymin + ymax) / 2.0) / height
                    box_w = (xmax - xmin) / width
                    box_h = (ymax - ymin) / height
                    yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")
                except:
                    continue
        with open(txt_file_path, "w") as f:
            f.write("\n".join(yolo_lines))

target_dir = "/content/UDAO-LASFNet/dataset/dronevehicle_new"
print("Mengonversi XML ke format TXT YOLO...")
for mode in ['train', 'val', 'test']:
    convert_dronevehicle_xml_to_yolo(f"{target_dir}/data_rgb/{mode}/labels", f"{target_dir}/data_rgb/{mode}/labels")
    convert_dronevehicle_xml_to_yolo(f"{target_dir}/data_ir/{mode}/labels", f"{target_dir}/data_ir/{mode}/labels")
    !find {target_dir}/data_rgb/{mode}/labels -name "*.xml" -delete 2>/dev/null || true
    !find {target_dir}/data_ir/{mode}/labels -name "*.xml" -delete 2>/dev/null || true

def synchronize_dataset(base_dir):
    for mode in ['train', 'val', 'test']:
        print(f"\n--- Sinkronisasi Dataset: {mode.upper()} ---")
        rgb_img_dir = os.path.join(base_dir, f"data_rgb/{mode}/images")
        rgb_lbl_dir = os.path.join(base_dir, f"data_rgb/{mode}/labels")
        ir_img_dir = os.path.join(base_dir, f"data_ir/{mode}/images")
        ir_lbl_dir = os.path.join(base_dir, f"data_ir/{mode}/labels")

        rgb_imgs = {os.path.splitext(f)[0] for f in os.listdir(rgb_img_dir) if f.lower().endswith(('.jpg', '.jpeg'))} if os.path.exists(rgb_img_dir) else set()
        rgb_lbls = {os.path.splitext(f)[0] for f in os.listdir(rgb_lbl_dir) if f.lower().endswith('.txt')} if os.path.exists(rgb_lbl_dir) else set()
        ir_imgs = {os.path.splitext(f)[0] for f in os.listdir(ir_img_dir) if f.lower().endswith(('.jpg', '.jpeg'))} if os.path.exists(ir_img_dir) else set()
        ir_lbls = {os.path.splitext(f)[0] for f in os.listdir(ir_lbl_dir) if f.lower().endswith('.txt')} if os.path.exists(ir_lbl_dir) else set()

        valid_keys = rgb_imgs.intersection(rgb_lbls).intersection(ir_imgs).intersection(ir_lbls)
        print(f"Jumlah sampel sinkron sempurna: {len(valid_keys)}")

        def clean_unpaired(directory, valid_set, exts):
            if not os.path.exists(directory): return
            for f in os.listdir(directory):
                name, ext = os.path.splitext(f)
                if ext.lower() in exts and name not in valid_set:
                    try: os.remove(os.path.join(directory, f))
                    except: pass

        clean_unpaired(rgb_img_dir, valid_keys, ['.jpg', '.jpeg', '.png'])
        clean_unpaired(rgb_lbl_dir, valid_keys, ['.txt'])
        clean_unpaired(ir_img_dir, valid_keys, ['.jpg', '.jpeg', '.png'])
        clean_unpaired(ir_lbl_dir, valid_keys, ['.txt'])

synchronize_dataset(target_dir)
!find {target_dir} -name "*.cache" -delete 2>/dev/null || true
print("Dataset terverifikasi, sinkron, dan bersih!")

### **LANGKAH 5: Konfigurasi File YAML Dataset (`data/dronevehicle_new.yaml`) & Patch Kompatibilitas PyTorch 2.6+**

In [ ]:
import os

REPO_DIR = "/content/UDAO-LASFNet"

yaml_content = """# Dataset DroneVehicle untuk UDAO-LASFNet
path: /content/UDAO-LASFNet/dataset/dronevehicle_new
train_rgb: data_rgb/train/images
train_ir: data_ir/train/images
val_rgb: data_rgb/val/images
val_ir: data_ir/val/images
test_rgb: data_rgb/test/images
test_ir: data_ir/test/images

# Classes
nc: 5
names:
  0: car
  1: truck
  2: bus
  3: van
  4: freight car
"""

os.makedirs(f"{REPO_DIR}/data", exist_ok=True)
yaml_path = f"{REPO_DIR}/data/dronevehicle_new.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print(f"File konfigurasi YAML sukses dibuat di: {yaml_path}")

# Patch kompatibilitas PyTorch 2.6+ (weights_only=False) pada train.py dan val.py
header_patch = '''import sys
import torch
import utils.general as ug
ug.check_requirements = lambda *args, **kwargs: True
_orig_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)
torch.load = _patched_load
'''
for fname in ["train.py", "val.py"]:
    fpath = os.path.join(REPO_DIR, fname)
    with open(fpath, "r") as f:
        code = f.read()
    if "_patched_load" not in code:
        with open(fpath, "w") as f:
            f.write(header_patch + "\n" + code)
print("Patch kompatibilitas PyTorch 2.6+ siap!")

### **LANGKAH 6: Sanity Check & Profiling Arsitektur UDAO-LASFNet (< 8.0M Parameters)**

In [ ]:
%cd /content/UDAO-LASFNet
import torch
from models.yolo import Model

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = Model('models/udao_lasfnet.yaml', ch=3, nc=5).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal Parameters (Unfused): {total_params:,} ({total_params/1e6:.3f}M)")
assert total_params < 8_000_000, "Parameter melebihi batas 8.0M!"
print("Sanity Check LULUS: Model berada di bawah batas 8.0M parameter!")

### **LANGKAH 7: Mulai Proses Training UDAO-LASFNet (300 Epochs)**
- **Opsi A (Default)**: Full Proposed Model (`models/udao_lasfnet.yaml`)
- **Opsi B (Ablation Study)**: Ganti `--cfg` dengan `models/lasfnet_udao_only.yaml` atau `models/lasfnet_dfatm_only.yaml`

In [ ]:
%cd /content/UDAO-LASFNet

!python train.py \
  --weights "" \
  --cfg models/udao_lasfnet.yaml \
  --data data/dronevehicle_new.yaml \
  --hyp data/hyps/hyp.scratch.yaml \
  --epochs 300 \
  --batch-size 48 \
  --imgsz 640 \
  --device 0 \
  --workers 8 \
  --project "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/UDAO_LASFNet_DroneVehicle_Run" \
  --name "udao_lasfnet_300ep" \
  --exist-ok

### **LANGKAH 8: Evaluasi Model Terbaik (`best.pt`) pada Test Split DroneVehicle**

In [ ]:
%cd /content/UDAO-LASFNet

!python val.py \
  --data data/dronevehicle_new.yaml \
  --weights "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/UDAO_LASFNet_DroneVehicle_Run/udao_lasfnet_300ep/weights/best.pt" \
  --batch-size 32 \
  --imgsz 640 \
  --device 0 \
  --task test \
  --verbose